In [1]:
pip install python-telegram-bot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.5/669.5 kB 14.9 MB/s eta 0:00:00


In [16]:
#Libraries

import cv2
import numpy as np
from telegram.ext import *
from telegram import Update
from io import BytesIO
import tensorflow as tf
import asyncio

In [3]:
with open("/content/token.txt", "r") as x:
  TOKEN = x.read()

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255, x_test / 255

classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [5]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Conv2D(32, (3, 3), activation = "relu", input_shape = (32, 32, 3)))
model.add(tf.keras.layers.MaxPooling2D(2, 2))
model.add(tf.keras.layers.Conv2D(64, (3, 3), activation = "relu"))
model.add(tf.keras.layers.MaxPooling2D(2, 2))
model.add(tf.keras.layers.Conv2D(64, (3, 3), activation = "relu"))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(64, activation = "relu"))
model.add(tf.keras.layers.Dense(10, activation = "softmax"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Define the functions
async def start(update: Update, context: CallbackContext):
    await update.message.reply_text("Xoş gəlmisiniz!")

async def help(update: Update, context: CallbackContext):
    await update.message.reply_text("""
    /start - starts conversation
    /help - shows message
    /train - trains the network
    """)

async def train(update: Update, context: CallbackContext):
    await update.message.reply_text("Model is starting...")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10)
    model.save("cifar10.model")
    await update.message.reply_text("DONE!")

async def handle_message(update: Update, context: CallbackContext):
    await update.message.reply_text("Train and send a photo")

async def handle_photo(update: Update, context: CallbackContext):
    file = await context.bot.get_file(update.message.photo[-1].file_id)
    x = BytesIO(await file.download_as_bytearray())
    file_bytes = np.asarray(bytearray(x.read()), dtype=np.uint8)

    image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    image = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA)

    # Normalize image to range [0, 1] before prediction
    pred = model.predict(np.array([image / 255.0]))  # Normalizing pixel values

    await update.message.reply_text(f"If I'm not mistaken, it is a {classes[np.argmax(pred)]}")

updater = Application.builder().token(TOKEN).build()

updater.add_handler(CommandHandler("start", start))
updater.add_handler(CommandHandler("help", help))
updater.add_handler(CommandHandler("train", train))

updater.add_handler(MessageHandler(filters.TEXT, handle_message))
updater.add_handler(MessageHandler(filters.PHOTO, handle_photo))

if __name__ == '__main__':
    import nest_asyncio
    nest_asyncio.apply()  # Apply nest_asyncio to avoid conflict in environments like Jupyter
    updater.run_polling()

/usr/lib/python3.11/importlib/__init__.py:126: RuntimeWarning: coroutine 'Application.shutdown' was never awaited
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/lib/python3.11/importlib/__init__.py:126: RuntimeWarning: coroutine 'Application.initialize' was never awaited
  return _bootstrap._gcd_import(name[level:], package, level)


Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 48ms/step - accuracy: 0.3601 - loss: 1.7269 - val_accuracy: 0.5561 - val_loss: 1.2515
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 49ms/step - accuracy: 0.5764 - loss: 1.1911 - val_accuracy: 0.6162 - val_loss: 1.0689
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 75s 48ms/step - accuracy: 0.6469 - loss: 1.0058 - val_accuracy: 0.6567 - val_loss: 0.9710
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 48ms/step - accuracy: 0.6850 - loss: 0.8947 - val_accuracy: 0.6861 - val_loss: 0.8930
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 50ms/step - accuracy: 0.7184 - loss: 0.7984 - val_accuracy: 0.6922 - val_loss: 0.8973
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - accuracy: 0.7442 - loss: 0.7375 - val_accuracy: 0.6979 - val_loss: 0.8711
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 79s 47ms/step - accuracy: 0.7607 - loss: 0.6869 - val_accuracy: 0.7015 - val_loss: 0.8772
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 81s 52ms/step - accuracy: 0.7714 -

ERROR:telegram.ext.Application:No error handlers are registered, logging exception.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/telegram/ext/_application.py", line 1325, in process_update
    await coroutine
  File "/usr/local/lib/python3.11/dist-packages/telegram/ext/_handlers/basehandler.py", line 158, in handle_update
    return await self.callback(update, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-22-e5c5e118ac89>", line 16, in train
    model.save("cifar10.model")
  File "/usr/local/lib/python3.11/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_api.py", line 114, in save_model
    raise ValueError(
ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `mo

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
